# Notebook to clean crash data

In [ ]:
# install packages if necessary
%pip install -r "../requirements.txt"

In [184]:
import pandas as pd
import geopandas as gpd
import zipfile

# Function to load data
- Loads in collision data from [San Francisco's Open Data Portal](https://data.sfgov.org/Public-Safety/Map-of-Traffic-Crashes-Resulting-in-Injuries/kn4t-hihx)

- Currently data in the range between January 1st, 2005 to December 31st, 2025 is available.
- For our objectives, we downloaded information on collisions recorded between December of 2006 to June of 2016 to have a period consistent with the data from the Stanford Open Policing Project. 

In [185]:
def load_data(path, lat = None, long = None, geospatial = False):
    # path is the string of the path to datafile
    # lat/long are strings for the name of the latitude and longitude columns in the datafile
    # set geospatial to true if trying to extract xy coordinates from a csv file
    
    # unzip data if name ends with .zip
    if(path[-3:]=="zip"):
        with zipfile.ZipFile(path) as z:
            csv_file = [f for f in z.namelist() if f.endswith(".csv")][0]
            df = pd.read_csv(z.open(csv_file))
        print("Unzipped")
    # read in regularly for non zipped files
    else:
        # split into last file name after /
        if("geojson" in path.rsplit('/', 1)[-1]):
            df = gpd.read_file(path)
        else:
            df = pd.read_csv(path)

    # convert to a geospatial dataframe to extract x,y coordinates
    if(geospatial):
        df = gpd.GeoDataFrame(
            df,
            geometry=gpd.points_from_xy(df[long], df[lat]),
            # crs set for lat/long data
            crs="EPSG:4326" 
        )

        # Print coordinate reference system to verify correct locations
        print("Data CRS:", df.crs)
    
    return df

# Filter by location
- Since we are only interested in collisions occuring in San Francisco county, we filter out any data from outside the county boundaries

In [186]:
def filter_data_loc(df, county_df, counties_of_interest):
    # df is data to filter
    # county_df is a geospatial dataframe containing polygons that define the county of interest
    # counties of interest is a list of strings with each string corresponding to a county in the county_df that is necessary to keep

    # filter points to within bounaries of county polygon
    boundaries = county_df[county_df['county'].isin(counties_of_interest)]
    filtered_df = gpd.sjoin(df, boundaries, predicate="within")
     
    return filtered_df

If necessary, change the local path names `crash_path` and `county_path`, counties of interest (`counties_of_interest`), and column names for data (`lat` and `long` arguments in the `load_data` function)

In [187]:
# change path names if necessary
crash_path = r"../data/collisions_raw.csv.zip"
county_path = r"../data/Bay_Area_County_Polygons.geojson"

# change to different counties of interest if necessary
counties_of_interest = ['San Francisco']

# load crash, county data
# change latitude and longitude column names if necessary
crash_df = load_data(crash_path, lat = "tb_latitude", long = "tb_longitude", geospatial = True)
county_df = load_data(county_path)

# filter to coordinates within relevant counties
crash_df = filter_data_loc(crash_df, county_df, counties_of_interest)


Unzipped
Data CRS: EPSG:4326


# Filter by time
- Since we are only interested in collisions that overlap with the Vision Zero data, we filter data to keep observations between the period of January of 2010 to June of 2016

In [188]:
def filter_data_time(df, dt_col, start, stop):
    # check values are in datetime format
    df[dt_col] = pd.to_datetime(df[dt_col])

    start = pd.to_datetime(start)
    stop = pd.to_datetime(stop)
    # filter
    df = df[df[dt_col].between(start, stop)]

    return df


If necessary, change the dates to filter

In [189]:
crash_df = filter_data_time(crash_df, 'collision_datetime', '1-01-2010', '06-01-2016')

/var/folders/6l/_5f2v6_x5hj1cd_lrwrm8k9r0000gn/T/ipykernel_29887/3471573753.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[dt_col] = pd.to_datetime(df[dt_col])


# Features and feature engineering
- For analyzing collision data, we are interested in certain features
- Here we restrict data to our relevant predictor and outcome variables

In [190]:
var = ['tb_latitude', 'tb_longitude', 
 'collision_datetime',
 'weather_1', 'lighting', 'road_surface', 'road_cond_1',
 'collision_severity', 'type_of_collision', 'mviw', 'ped_action', 
 'number_killed', 'number_injured']

crash_df = crash_df[var]

# get number of rows and columns for observations
crash_df.shape

(19818, 13)

In [191]:
# check distinct values for columns that aren't time/space
cols_to_exclude = ['tb_longitude', 'tb_latitude', 'collision_datetime']
for column_name in crash_df.loc[:, ~crash_df.columns.isin(cols_to_exclude)]:
    print(f"Unique Values for {column_name}")
    print(pd.unique(crash_df[column_name].values.ravel()))
    print()

Unique Values for weather_1
['Clear' 'Raining' 'Cloudy' 'Other' 'Fog' 'Not Stated' 'Wind' 'Snowing']

Unique Values for lighting
['Dark - Street Lights' 'Daylight' 'Dusk - Dawn' 'Dark - No Street Lights'
 'Not Stated' 'Dark - Street Lights Not Functioning']

Unique Values for road_surface
['Dry' 'Wet' 'Not Stated' 'Snowy or Icy' 'Slippery']

Unique Values for road_cond_1
['No Unusual Condition' 'Other' 'Construction or Repair Zone'
 'Holes, Deep Ruts' 'Not Stated' 'Loose Material on Roadway'
 'Obstruction on Roadway' 'Reduced Roadway Width' 'Flooded']

Unique Values for collision_severity
['Injury (Complaint of Pain)' 'Injury (Other Visible)' 'Injury (Severe)'
 'Fatal']

Unique Values for type_of_collision
['Rear End' 'Overturned' 'Vehicle/Pedestrian' 'Hit Object' 'Sideswipe'
 'Broadside' 'Other' 'Not Stated' 'Head-On']

Unique Values for mviw
['Other Motor Vehicle' 'Fixed Object' 'Pedestrian' 'Train' 'Non-Collision'
 'Bicycle' 'Motor Vehicle on Other Roadway' 'Parked Motor Vehicle'
 '

- We also want to remove NA values that cannot be imputed
- Based on the distinct values, 'Not Stated' also acts as an NA value here

In [192]:
# drop nas and unstated values
crash_df = crash_df.dropna()
crash_df = crash_df[~crash_df.isin(['Not Stated']).any(axis=1)]

# check dimensions after removing na's
crash_df.shape

(18652, 13)

- Now we want to add a feature to flag whether an accident occurs in a Community of concern.
- We load in data from the [Metropolitan Transportation Commission](https://opendata-mtc.opendata.arcgis.com/datasets/MTC::equity-priority-communities-plan-bay-area-2040/about) that contains boundaries for communities of concern across the Bay Area.
- Here we alter the Equal Priority Community (epc, synonymous with Community of Concern) Class Flag to Change NA values to 'None' if an area is not an epc
- We also add data on the demographics of the community including the proportion of the population who are people of color, disabled, or 75+.

In [194]:
# Load data for communities of concern
concern_path = r"../data/communities_of_concern.geojson"
concern_df = load_data(concern_path)

# change epc_class to make NA = None
concern_df['epc_class'] = concern_df['epc_class'].replace({'NA':'None'})

# add features of interest to collision events
concern_features = ['pct_over75', 'pct_poc', 'pct_disab', 'epc_class', 'geometry']
concern_df = concern_df[concern_features]

# check that collision data is in geopandas format
crash_df = gpd.GeoDataFrame(
    crash_df,
    geometry=gpd.points_from_xy(crash_df.tb_longitude, crash_df.tb_latitude),
    crs="EPSG:4326"
)

# add community of concern data to crash observations
crash_concern = gpd.sjoin(crash_df, concern_df, how="inner", predicate="within")
# drop unnecessary columns
unnecessary_cols = ['geometry', 'index_right']
crash_concern = crash_concern.drop(columns = unnecessary_cols)
crash_concern.head()

,tb_latitude,tb_longitude,collision_datetime,weather_1,lighting,road_surface,road_cond_1,collision_severity,type_of_collision,mviw,ped_action,number_killed,number_injured,pct_over75,pct_poc,pct_disab,epc_class
9241,37.782932,-122.420856,2010-01-01 02:06:00,Clear,Dark - Street Lights,Dry,No Unusual Condition,Injury (Complaint of Pain),Rear End,Other Motor Vehicle,No Pedestrian Involved,0.0,2,0.074700,0.638495,0.228190,High
9242,37.725067,-122.429108,2010-01-01 02:38:00,Clear,Dark - Street Lights,Dry,No Unusual Condition,Injury (Complaint of Pain),Overturned,Fixed Object,No Pedestrian Involved,0.0,1,0.044399,0.900186,0.084562,High
9243,37.734797,-122.390694,2010-01-01 16:30:00,Clear,Daylight,Dry,No Unusual Condition,Injury (Other Visible),Vehicle/Pedestrian,Pedestrian,Crossing in Crosswalk at Intersection,0.0,1,0.035538,0.881149,0.101791,Higher
9244,37.789592,-122.420488,2010-01-01 18:59:00,Raining,Dark - Street Lights,Wet,No Unusual Condition,Injury (Complaint of Pain),Vehicle/Pedestrian,Pedestrian,Crossing in Crosswalk at Intersection,0.0,1,0.078134,0.419996,0.176776,None
9245,37.723900,-122.401979,2010-01-02 16:50:00,Clear,Dusk - Dawn,Dry,No Unusual Condition,Injury (Other Visible),Hit Object,Fixed Object,No Pedestrian Involved,0.0,1,0.057739,0.858806,0.070839,High


In [195]:
### export new cleaned dataset
crash_concern.to_csv(r"../data/collisions_clean.csv")